## Imports e configuração

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import pyarrow.dataset as ds
import numpy as np
import os
import gc
import pyarrow.dataset as ds
import glob
from scipy.stats import chisquare



os.makedirs("../outputs/processed", exist_ok=True)
os.makedirs("../outputs/figures", exist_ok=True)


## Leitura dos dados processados

In [3]:
deng = glob.glob("../data/raw/dengue_sinan/*.parquet")
chik = glob.glob("../data/raw/chikungunya_sinan/*.parquet")
zika = glob.glob("../data/raw/zika_sinan/*.parquet")
pop = "../data/raw/municipios_ibge/POP2025_20260113.xls"

pd.set_option('display.max_columns', None)

Capturar Registros Duplicados


In [ ]:
for i, arquivo in enumerate(deng[:],):

    print(f"\nProcessando {i + 1}/{len(deng)}")
    print(f"Arquivo: {arquivo}")

    df = pd.read_parquet(arquivo)

    dup = df.duplicated(keep=False)
    total = len(df)
    duplicados = dup.sum()
    unicos = (~dup).sum()
    percentual = duplicados / total * 100

    print(f"Total de registros: {total}")
    print(f"Registros duplicados: {duplicados}")
    print(f"Registros únicos: {unicos}")
    print(f"Percentual duplicado: {percentual:.2f}%")

    del dup
    del df

    gc.collect()


Proporção de Casos por Gênero

In [ ]:
DOENCAS = {
    "Dengue":      glob.glob("../data/raw/dengue_sinan/*.parquet"),
    "Chikungunya": glob.glob("../data/raw/chikungunya_sinan/*.parquet"),
    "Zika":        glob.glob("../data/raw/zika_sinan/*.parquet"),
}

for nome, arquivos in DOENCAS.items():
    serie = pd.concat([
        ds.dataset(a, format="parquet").to_table(columns=["CS_SEXO"]).to_pandas()["CS_SEXO"]
        for a in arquivos
    ], ignore_index=True)

    counts = serie.value_counts(dropna=False)
    total = len(serie)

    print(f"\n=== {nome} ===")
    for k, v in counts.items():
        print(f"  {k or 'NaN'}: {v:,} ({v/total*100:.1f}%)")

    mf = serie[serie.isin(["M", "F"])].value_counts().reindex(["M", "F"], fill_value=0)
    chi2, p = chisquare(mf.values)
    razao = mf["F"] / mf["M"] if mf["M"] > 0 else float("inf")
    print(f"  → F/M={razao:.2f}  chi2={chi2:.1f}  p={p:.2e}")

In [ ]:
def decode_idade(serie):
    s = pd.to_numeric(serie, errors="coerce").dropna().astype(int).reset_index(drop=True)
    unidade = s // 1000     
    valor   = s % 1000         
    idade = pd.Series(np.nan, index=s.index)
    idade[unidade == 4] = valor[unidade == 4]
    idade[unidade == 3] = valor[unidade == 3] / 12
    idade[unidade == 2] = valor[unidade == 2] / 365
    idade[unidade == 1] = valor[unidade == 1] / 8760
    return idade

DOENCAS = {
    "Dengue": glob.glob("../data/raw/dengue_sinan/*.parquet")
}

for nome, arquivos in DOENCAS.items():
    serie = pd.concat([
        ds.dataset(a, format="parquet").to_table(columns=["NU_IDADE_N"]).to_pandas()["NU_IDADE_N"]
        for a in arquivos], ignore_index=True)

    idade = decode_idade(serie)
    total_raw = len(serie)
    validos   = idade.dropna()

    Q1, Q3 = validos.quantile(0.25), validos.quantile(0.75)
    IQR = Q3 - Q1
    limite_sup = Q3 + 1.5 * IQR
    limite_inf = Q1 - 1.5 * IQR  
    outliers = validos[(validos < limite_inf) | (validos > limite_sup)]

    print(f"\n=== {nome} ===")
    print(f"  Total registros : {total_raw:,}")
    print(f"  Idade válida    : {len(validos):,} ({len(validos)/total_raw*100:.1f}%)")
    print(f"  Nulos/inválidos : {total_raw - len(validos):,}")
    print(f"  Média   : {validos.mean():.1f} anos")
    print(f"  Mediana : {validos.median():.1f} anos")
    print(f"  Desvio  : {validos.std():.1f} anos")
    print(f"  Min     : {validos.min():.2f}  |  Max: {validos.max():.1f}")
    print(f"  Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}")
    print(f"  Limite outlier  : < {max(limite_inf,0):.1f} ou > {limite_sup:.1f} anos")
    print(f"  Outliers        : {len(outliers):,} ({len(outliers)/len(validos)*100:.2f}%)")
    print(f"  Faixa outliers  : {outliers.min():.1f} – {outliers.max():.1f} anos")
    print(f"  Assimetria      : {validos.skew():.3f}")

In [ ]:
def decode_idade(serie):
    s = serie.dropna().astype(str).str.strip()
    idade = pd.Series(np.nan, index=s.index)

    mask_antigo = s.str.match(r'^[A-Za-z]\d+$')
    if mask_antigo.any():
        sa      = s[mask_antigo]
        unidade = sa.str[0].str.upper()
        valor   = pd.to_numeric(sa.str[1:], errors="coerce")
        idade[mask_antigo & (unidade == "A")] = valor[mask_antigo & (unidade == "A")]
        idade[mask_antigo & (unidade == "M")] = valor[mask_antigo & (unidade == "M")] / 12
        idade[mask_antigo & (unidade == "D")] = valor[mask_antigo & (unidade == "D")] / 365
        idade[mask_antigo & (unidade == "H")] = valor[mask_antigo & (unidade == "H")] / 8760

    mask_novo = s.str.match(r'^\d{4}$')
    if mask_novo.any():
        sn      = pd.to_numeric(s[mask_novo], errors="coerce").dropna().astype(int)
        unidade = sn // 1000
        valor   = sn % 1000
        idade[sn.index[unidade == 4]] = valor[unidade == 4].values
        idade[sn.index[unidade == 3]] = valor[unidade == 3].values / 12
        idade[sn.index[unidade == 2]] = valor[unidade == 2].values / 365
        idade[sn.index[unidade == 1]] = valor[unidade == 1].values / 8760

    return idade


def carregar_idade(arquivos):
    partes = []
    for arq in sorted(arquivos):
        tbl  = ds.dataset(arq, format="parquet").to_table()
        cols = tbl.schema.names
        if "NU_IDADE_N" in cols:
            col = "NU_IDADE_N"
        elif "NU_IDADE" in cols:
            col = "NU_IDADE"
        else:
            continue
        partes.append(tbl.select([col]).to_pandas()[col])
    return pd.concat(partes, ignore_index=True)


todos = sorted(glob.glob("../data/raw/dengue_sinan/*.parquet"))

ate23 = [a for a in todos if not a.endswith("BR24.parquet")]
s24   = [a for a in todos if a.endswith("BR24.parquet")]

print("Carregando até 2023...")
serie_ate23 = carregar_idade(ate23)

print("Carregando 2024...")
serie_24 = carregar_idade(s24)

print("Calculando...")
serie   = pd.concat([serie_ate23, serie_24], ignore_index=True)
del serie_ate23, serie_24  # libera RAM

idade   = decode_idade(serie)
del serie

total_raw = len(idade) + idade.isna().sum()  # aproximado
validos   = idade.dropna()

Q1, Q3    = validos.quantile(0.25), validos.quantile(0.75)
IQR       = Q3 - Q1
limite_sup = Q3 + 1.5 * IQR
limite_inf = Q1 - 1.5 * IQR
outliers  = validos[(validos < limite_inf) | (validos > limite_sup)]

print(f"\n=== Dengue ===")
print(f"  Idade válida    : {len(validos):,}")
print(f"  Média   : {validos.mean():.1f} anos")
print(f"  Mediana : {validos.median():.1f} anos")
print(f"  Desvio  : {validos.std():.1f} anos")
print(f"  Assimetria      : {validos.skew():.3f}")
print(f"  Min     : {validos.min():.2f}  |  Max: {validos.max():.1f}")
print(f"  Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}")
print(f"  Limite outlier  : < {max(limite_inf,0):.1f} ou > {limite_sup:.1f} anos")
print(f"  Outliers        : {len(outliers):,} ({len(outliers)/len(validos)*100:.2f}%)")
if len(outliers):
    print(f"  Faixa outliers  : {outliers.min():.1f} – {outliers.max():.1f} anos")

In [ ]:
MAPA_RACA = {
    "1": "Branca",
    "2": "Preta",
    "3": "Amarela",
    "4": "Parda",
    "5": "Indígena",
    "9": "Ignorado",
}

DOENCAS = {
    "Chikungunya": glob.glob("../data/raw/chikungunya_sinan/*.parquet"),
    "Zika":        glob.glob("../data/raw/zika_sinan/*.parquet"),
}

for nome, arquivos in DOENCAS.items():
    serie = pd.concat([
        ds.dataset(a, format="parquet").to_table(columns=["CS_RACA"]).to_pandas()["CS_RACA"]
        for a in arquivos
    ], ignore_index=True)

    serie = serie.astype(str).str.strip()
    total = len(serie)
    counts = serie.map(MAPA_RACA).fillna("Nulo/Outro").value_counts()

    print(f"\n=== {nome} ===")
    for cat, n in counts.items():
        print(f"  {cat}: {n:,} ({n/total*100:.1f}%)")

In [ ]:
MAPA_ESCOL = {
    "0":  "Analfabeto",
    "1":  "1ª a 4ª série incompleta (EF)",
    "2":  "4ª série completa (EF)",
    "3":  "5ª a 8ª série incompleta (EF)",
    "4":  "Ensino fundamental completo",
    "5":  "Ensino médio incompleto",
    "6":  "Ensino médio completo",
    "7":  "Educação superior incompleta",
    "8":  "Educação superior completa",
    "9":  "Ignorado",
    "10": "Não se aplica",
}

DOENCAS = {
    "Chikungunya": glob.glob("../data/raw/chikungunya_sinan/*.parquet"),
    "Zika":        glob.glob("../data/raw/zika_sinan/*.parquet"),
}

for nome, arquivos in DOENCAS.items():
    serie = pd.concat([
        ds.dataset(a, format="parquet").to_table(columns=["CS_ESCOL_N"]).to_pandas()["CS_ESCOL_N"]
        for a in arquivos
    ], ignore_index=True)

    serie = serie.astype(str).str.strip()
    total = len(serie)
    counts = serie.map(MAPA_ESCOL).fillna("Nulo/Outro").value_counts()

    print(f"\n=== {nome} ===")
    for cat, n in counts.items():
        print(f"  {cat}: {n:,} ({n/total*100:.1f}%)")

## Contagem total de casos

In [ ]:
contagem= 0
contagem2 = 0
contagem3 = 0
for i in deng:
    dataset = ds.dataset(i, format="parquet")
    contagem += dataset.count_rows()


for i in chik:
    dataset = ds.dataset(i, format="parquet")
    contagem2 += dataset.count_rows()

for i in zika:
    dataset = ds.dataset(i, format="parquet")
    contagem3 += dataset.count_rows()

print(f"Dengue: {contagem} Chik: {contagem2} zika: {contagem3}")

Dengue: 32994356 Chik: 2437142,zika 604792


## Contagem de nulos

In [9]:
def analisar_nulos(*base, coluna):

    total_geral = 0
    nulos_geral = 0
    soma_ponderada = 0

    for indice, arquivo in enumerate(base, 1):

        nome_arquivo = os.path.basename(arquivo)

        print(f"\n[{indice}/{len(base)}] Processando: {nome_arquivo}")

        try:
            df = pd.read_parquet(
                arquivo,
                engine="pyarrow",
                columns=[coluna]
            )

        except Exception:
            print(f"  → Coluna '{coluna}' não encontrada. Pulando...")
            continue

        df = df.replace(r"^\s*\**$", np.nan, regex=True)

        total_linhas = len(df)

        nulos = df[coluna].isnull().sum()

        porcentagem = (
            0
            if total_linhas == 0
            else (nulos / total_linhas) * 100
        )

        total_geral += total_linhas
        nulos_geral += nulos

        soma_ponderada += porcentagem * total_linhas

        print(f"  → Registros: {total_linhas:,}")
        print(f"  → Nulos: {nulos:,}")
        print(f"  → % nulos: {porcentagem:.2f}%")

        del df
        gc.collect()

    porcentagem_geral = (
        0
        if total_geral == 0
        else (nulos_geral / total_geral) * 100
    )

    media_ponderada = (
        0
        if total_geral == 0
        else soma_ponderada / total_geral
    )

    df_final = pd.DataFrame([{
        "coluna": coluna,
        "total_registros": total_geral,
        "total_nulos": nulos_geral,
        "percentual_nulos_geral": round(porcentagem_geral, 2),
        "media_ponderada_%": round(media_ponderada, 2)
    }])

    print("\n" + "=" * 60)
    print("RESULTADO GERAL")
    print("=" * 60)

    print(f"Coluna: {coluna}")
    print(f"Total de registros: {total_geral:,}")
    print(f"Total de nulos: {nulos_geral:,}")
    print(f"Percentual geral de nulos: {porcentagem_geral:.2f}%")
    print(f"Média ponderada das porcentagens: {media_ponderada:.2f}%")

    print("=" * 60)

    return df_final
analisar_nulos(
    *deng,
    coluna="ID_MUNICIP"
)


[1/27] Processando: DENGBR00.parquet
  → Registros: 172,855
  → Nulos: 0
  → % nulos: 0.00%

[2/27] Processando: DENGBR01.parquet
  → Registros: 488,590
  → Nulos: 0
  → % nulos: 0.00%

[3/27] Processando: DENGBR02.parquet
  → Registros: 897,093
  → Nulos: 0
  → % nulos: 0.00%

[4/27] Processando: DENGBR03.parquet
  → Registros: 416,609
  → Nulos: 0
  → % nulos: 0.00%

[5/27] Processando: DENGBR04.parquet
  → Registros: 136,867
  → Nulos: 0
  → % nulos: 0.00%

[6/27] Processando: DENGBR05.parquet
  → Registros: 261,501
  → Nulos: 1
  → % nulos: 0.00%

[7/27] Processando: DENGBR06.parquet
  → Registros: 411,022
  → Nulos: 0
  → % nulos: 0.00%

[8/27] Processando: DENGBR07.parquet
  → Registros: 717,097
  → Nulos: 0
  → % nulos: 0.00%

[9/27] Processando: DENGBR08.parquet
  → Registros: 919,324
  → Nulos: 0
  → % nulos: 0.00%

[10/27] Processando: DENGBR09.parquet
  → Registros: 600,658
  → Nulos: 0
  → % nulos: 0.00%

[11/27] Processando: DENGBR10.parquet
  → Registros: 1,381,254
  → N

,coluna,total_registros,total_nulos,percentual_nulos_geral,media_ponderada_%
0,ID_MUNICIP,32994356,49,0.0,0.0


In [ ]:
def analisar_nulos(base, outputname):
    resultados = []

    for indice in base:
        try:
            df = pd.read_parquet(indice, engine="pyarrow",columns=["ID_OCUPA_N"])
        except Exception as e:
            print(f"Erro ao ler {indice}")
            continue
        df = df.replace(r"^\s*$", np.nan, regex=True)
        total_linhas = len(df)

        linha = {"parquet": os.path.basename(indice)}

        for col in df.columns:
            nulos = df[col].isnull().sum()
            porcentagem = 0 if total_linhas == 0 else (nulos / total_linhas) * 100
            linha[f"{col}_%"] = round(porcentagem, 2)

        resultados.append(linha)
        del df
        gc.collect()

    df_final = pd.DataFrame(resultados)
    print(df_final)


analisar_nulos(zika, "analise_nulos_dengue")


## Diagnóstico de ID de município

In [5]:
def diagnostico_id_municip(base, nome_base):
    contador_total = pd.Series(dtype=int)

    for arquivo in base:
        df = pd.read_parquet(arquivo, columns=["ID_MUNICIP"])
        df["ID_MUNICIP"] = df["ID_MUNICIP"].astype(str).str.strip()
        contagem = df["ID_MUNICIP"].str.len().value_counts()
        contador_total = contador_total.add(contagem, fill_value=0)
        del df

    contador_total = contador_total.sort_index()
    print(f"\nDiagnóstico ID_MUNICIP — {nome_base}")
    print(contador_total)
    return contador_total


diag_deng = diagnostico_id_municip(deng, "Dengue")
diag_chik = diagnostico_id_municip(chik, "Chikungunya")

# salva diagnóstico combinado
df_diag = pd.DataFrame({
    "comprimento_codigo": diag_deng.index.union(diag_chik.index),
})
df_diag["dengue"]      = df_diag["comprimento_codigo"].map(diag_deng).fillna(0).astype(int)
df_diag["chikungunya"] = df_diag["comprimento_codigo"].map(diag_chik).fillna(0).astype(int)

caminho_diag = "../outputs/processed/diagnostico_id_municip.csv"
df_diag.to_csv(caminho_diag, index=False)
print(f"\nSalvo: {caminho_diag}")



Diagnóstico ID_MUNICIP — Dengue
ID_MUNICIP
0          49.0
1           3.0
3           1.0
5           1.0
6    30440880.0
7     2784528.0
dtype: float64

Diagnóstico ID_MUNICIP — Chikungunya
ID_MUNICIP
6    2376910.0
dtype: float64

Salvo: ../outputs/processed/diagnostico_id_municip.csv


## Mapeamento dos municípios

In [ ]:
import pandas as pd


def processar_municipios(base, nome):

    caminhotab = "../data/raw/municipios_ibge/Tabela_municipios.xls"

    base = sorted(base)

    df_mun = pd.read_excel(caminhotab, skiprows=6)

    df_mun.columns = df_mun.columns.str.strip()

    COL_COD = "Código Município Completo"
    COL_NOME = "Nome_Município"

    df_mun["ID_MUNICIP"] = df_mun[COL_COD].astype(str).str.zfill(7)
    df_mun["COD6"] = df_mun["ID_MUNICIP"].str[:6]

    mapa_mun = dict(zip(df_mun["COD6"], df_mun[COL_NOME]))

    regioes = {
        "1": "Norte",
        "2": "Nordeste",
        "3": "Sudeste",
        "4": "Sul",
        "5": "Centro-Oeste"
    }

    lista = []
    total_lidos = 0

    for i in base:

        print(f"Processando: {i}")

        df = pd.read_parquet(
            i,
            columns=[
                "NU_ANO",
                "ID_MUNICIP",
                "DT_SIN_PRI"
            ]
        )

        df["ID_MUNICIP"] = df["ID_MUNICIP"].astype(str).str.strip()

        total_lidos += len(df)

        df = df[df["ID_MUNICIP"].str.len().isin([6, 7])]

        df["COD6"] = df["ID_MUNICIP"].str[:6]

        df["Município"] = df["COD6"].map(mapa_mun)

        df["Região"] = df["COD6"].str[0].map(regioes)

        df = df.dropna(subset=["Município"])

        lista.append(df)

        del df

    df_final = pd.concat(lista, ignore_index=True)

    caminho_saida = f"../outputs/processed/{nome}_municipios.parquet"

    df_final.to_parquet(caminho_saida, index=False)

    return df_final


processar_municipios(deng, "dengue")
processar_municipios(chik, "chik")

In [ ]:
import pandas as pd

deng_casos = pd.read_parquet("../outputs/processed/dengue_municipios.parquet")
chik_casos = pd.read_parquet("../outputs/processed/chik_municipios.parquet")


def gerar_incidencia(df_municipios, arquivo_pop, nome, pop_min=50000):

    casos = (
        df_municipios
        .groupby(["Município", "NU_ANO"])
        .size()
        .reset_index(name="CASOS")
    )

    casos["Município"] = (
        casos["Município"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    pop = pd.read_excel(
        arquivo_pop,
        skiprows=1,
        sheet_name="Municípios"
    )

    pop.columns = pop.columns.str.strip()

    pop["Município"] = (
        pop["NOME DO MUNICÍPIO"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    pop["POPULACAO"] = pd.to_numeric(
        pop["POPULAÇÃO ESTIMADA"],
        errors="coerce"
    )

    incidencia = casos.merge(
        pop[["Município", "POPULACAO"]],
        on="Município",
        how="left"
    )

    incidencia = incidencia.dropna(
        subset=["POPULACAO"]
    )

    incidencia = incidencia[
        incidencia["POPULACAO"] >= pop_min
    ]

    incidencia["INCIDENCIA_100K"] = (
        incidencia["CASOS"]
        / incidencia["POPULACAO"]
    ) * 100000

    incidencia = incidencia.sort_values(
        ["NU_ANO", "INCIDENCIA_100K"],
        ascending=[True, False]
    ).reset_index(drop=True)

    caminho_saida = (
        f"../outputs/processed/"
        f"incidencia_{nome}.parquet"
    )

    incidencia.to_parquet(
        caminho_saida,
        index=False
    )

    print(f"Arquivo salvo: {caminho_saida}")

    return incidencia

df_chik = gerar_incidencia(
    deng_casos,
    pop,
    "Dengue",
    50000
)

Arquivo salvo: ../outputs/processed/incidencia_chikungunya.parquet


## Sexo, raça e escolaridade — Dengue (vindo de p.py)

In [ ]:
import pyarrow.dataset as ds
import pandas as pd
import glob, os
from scipy.stats import chisquare

MAPA_SEXO = {"M": "Masculino", "F": "Feminino", "I": "Ignorado"}
MAPA_RACA = {"1": "Branca", "2": "Preta", "3": "Amarela", "4": "Parda", "5": "Indígena", "9": "Ignorado"}
MAPA_ESCOL = {
    "0": "Analfabeto", "1": "1ª a 4ª série incompleta (EF)", "2": "4ª série completa (EF)",
    "3": "5ª a 8ª série incompleta (EF)", "4": "Ensino fundamental completo",
    "5": "Ensino médio incompleto", "6": "Ensino médio completo",
    "7": "Educação superior incompleta", "8": "Educação superior completa",
    "9": "Ignorado", "10": "Não se aplica",
}

todos = sorted(glob.glob("../data/raw/dengue_sinan/*.parquet"))

sexo_list  = []
raca_list  = []
escol_list = []

for arq in todos:
    print(f"  Processando {os.path.basename(arq)}...", end="\r")
    tbl  = ds.dataset(arq, format="parquet").to_table()
    cols = tbl.schema.names

    # Sexo
    if "CS_SEXO" in cols:
        sexo_list.append(tbl.select(["CS_SEXO"]).to_pandas()["CS_SEXO"])

    # Raça
    if "CS_RACA" in cols:
        raca_list.append(tbl.select(["CS_RACA"]).to_pandas()["CS_RACA"])

    # Escolaridade — nome muda igual à idade
    if "CS_ESCOL_N" in cols:
        escol_list.append(tbl.select(["CS_ESCOL_N"]).to_pandas()["CS_ESCOL_N"])
    elif "CS_ESCOLAR" in cols:
        escol_list.append(tbl.select(["CS_ESCOLAR"]).to_pandas()["CS_ESCOLAR"])

    del tbl

print()

# ── SEXO ─────────────────────────────────────────────────────────────────────
sexo  = pd.concat(sexo_list, ignore_index=True).astype(str).str.strip()
total = len(sexo)
counts_sexo = sexo.map(MAPA_SEXO).fillna("Outro/Nulo").value_counts()

mf     = sexo[sexo.isin(["M", "F"])].value_counts().reindex(["M", "F"], fill_value=0)
chi2, p = chisquare(mf.values)
razao  = mf["F"] / mf["M"] if mf["M"] > 0 else float("inf")

print("=== SEXO ===")
for cat, n in counts_sexo.items():
    print(f"  {cat}: {n:,} ({n/total*100:.1f}%)")
print(f"  → F/M={razao:.2f}  chi2={chi2:.1f}  p={p:.2e}")

# ── RAÇA ──────────────────────────────────────────────────────────────────────
raca  = pd.concat(raca_list, ignore_index=True).astype(str).str.strip()
total = len(raca)
counts_raca = raca.map(MAPA_RACA).fillna("Nulo/Outro").value_counts()

print("\n=== RAÇA ===")
for cat, n in counts_raca.items():
    print(f"  {cat}: {n:,} ({n/total*100:.1f}%)")

# ── ESCOLARIDADE ──────────────────────────────────────────────────────────────
escol  = pd.concat(escol_list, ignore_index=True).astype(str).str.strip()
total  = len(escol)
counts_escol = escol.map(MAPA_ESCOL).fillna("Nulo/Outro").value_counts()

print("\n=== ESCOLARIDADE ===")
for cat, n in counts_escol.items():
    print(f"  {cat}: {n:,} ({n/total*100:.1f}%)")

## Estatísticas de idade incrementais — Dengue (vindo de script.py)

In [ ]:
import pyarrow.dataset as ds
import glob
import numpy as np

def decode_idade_np(arr):
    """Recebe array numpy de strings, retorna array de idades em anos (float)."""
    resultado = np.full(len(arr), np.nan)

    for i, v in enumerate(arr):
        if v is None:
            continue
        v = str(v).strip()

        # Formato antigo: letra + dígitos (A024, M06...)
        if v and v[0].isalpha():
            unidade = v[0].upper()
            try:
                valor = float(v[1:])
            except ValueError:
                continue
            if unidade == "A":   resultado[i] = valor
            elif unidade == "M": resultado[i] = valor / 12
            elif unidade == "D": resultado[i] = valor / 365
            elif unidade == "H": resultado[i] = valor / 8760

        # Formato novo: 4 dígitos numéricos (4024, 3006...)
        elif len(v) == 4 and v.isdigit():
            n = int(v)
            unidade = n // 1000
            valor   = n % 1000
            if unidade == 4:   resultado[i] = valor
            elif unidade == 3: resultado[i] = valor / 12
            elif unidade == 2: resultado[i] = valor / 365
            elif unidade == 1: resultado[i] = valor / 8760

    return resultado


# Acumuladores para estatísticas incrementais
n_total    = 0
n_validos  = 0
soma       = 0.0
soma_sq    = 0.0
soma_cb    = 0.0  # para assimetria
todos_vals = []   # só para mediana e outliers — lista de arrays pequenos

todos = sorted(glob.glob("../data/raw/dengue_sinan/*.parquet"))

for arq in todos:
    print(f"  Processando {arq.split('/')[-1]}...", end="\r")

    tbl  = ds.dataset(arq, format="parquet").to_table()
    cols = tbl.schema.names
    if "NU_IDADE_N" in cols:
        col = "NU_IDADE_N"
    elif "NU_IDADE" in cols:
        col = "NU_IDADE"
    else:
        continue

    arr  = tbl.select([col]).to_pandas()[col].to_numpy(dtype=str, na_value=None)
    del tbl

    idade = decode_idade_np(arr)
    del arr

    validos = idade[~np.isnan(idade)]
    del idade

    n_total   += len(validos) + np.isnan(validos).sum() if False else len(validos)
    n          = len(validos)
    n_validos += n
    print(f"  {arq.split('/')[-1]} → col={col}  n={n}")

    if n == 0:
        continue

    soma    += validos.sum()
    soma_sq += (validos ** 2).sum()
    soma_cb += (validos ** 3).sum()
    todos_vals.append(validos)
    del validos

print()

# ── Estatísticas finais ───────────────────────────────────────────────────────
media  = soma / n_validos
var    = (soma_sq / n_validos) - media ** 2
desvio = np.sqrt(var)

# Assimetria de Pearson (moment-based)
m3 = (soma_cb / n_validos) - 3 * media * var - media ** 3
assimetria = m3 / (desvio ** 3)

# Mediana e outliers: concatena tudo só aqui (uma vez)
print("Calculando mediana e outliers...")
tudo = np.concatenate(todos_vals)
del todos_vals

mediana = np.median(tudo)
Q1, Q3  = np.percentile(tudo, [25, 75])
IQR     = Q3 - Q1
lim_sup = Q3 + 1.5 * IQR
lim_inf = max(Q1 - 1.5 * IQR, 0)
outliers = tudo[(tudo < lim_inf) | (tudo > lim_sup)]
n_out    = len(outliers)
faixa    = (outliers.min(), outliers.max()) if n_out else (None, None)
print(f"  Min / Max       : {tudo.min():.1f} / {tudo.max():.1f} anos")
del tudo, outliers

print(f"\n=== Dengue ===")
print(f"  Idade válida    : {n_validos:,}")
print(f"  Média           : {media:.1f} anos")
print(f"  Mediana         : {mediana:.1f} anos")
print(f"  Desvio          : {desvio:.1f} anos")
print(f"  Assimetria      : {assimetria:.3f}")
print(f"  Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}")
print(f"  Limite outlier  : < {lim_inf:.1f} ou > {lim_sup:.1f} anos")
print(f"  Outliers        : {n_out:,} ({n_out/n_validos*100:.2f}%)")
if faixa[0] is not None:
    print(f"  Faixa outliers  : {faixa[0]:.1f} – {faixa[1]:.1f} anos")